# Wine Quality - Caderno Base de Analise

Este caderno resume e reproduz a preparacao principal do projeto Wine Quality. Ele usa os arquivos locais em `sample_data/`, combina vinhos tintos e brancos, calcula estatisticas descritivas, monta tabelas de frequencia e gera visualizacoes para apoiar respostas futuras.

## 1. Contexto

- Dataset: Wine Quality, UCI Machine Learning Repository.
- Amostras: vinho tinto e vinho branco do Vinho Verde portugues.
- Tamanho esperado: 1.599 tintos + 4.898 brancos = 6.497 observacoes.
- Variaveis: 11 fisico-quimicas, `quality` como nota sensorial ordinal e `wine_type` como tipo nominal.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "sample_data"
EXPORT_DIR = BASE_DIR / "export"
EXPORT_DIR.mkdir(exist_ok=True)

## 2. Carregamento dos dados locais

In [ ]:
red = pd.read_csv(DATA_DIR / "winequality-red.csv", sep=";")
white = pd.read_csv(DATA_DIR / "winequality-white.csv", sep=";")

red["wine_type"] = "red"
white["wine_type"] = "white"

df = pd.concat([red, white], ignore_index=True)

print("Dimensao:", df.shape)
display(df.head())
display(df["wine_type"].value_counts().rename("frequencia"))

## 3. Validacao rapida

A base original informa que nao ha valores ausentes. Esta celula confirma isso e mostra tipos de dados.

In [ ]:
display(df.dtypes.rename("tipo"))
display(df.isna().sum().rename("ausentes"))
display(df.describe().T)

## 4. Classificacao das variaveis

In [ ]:
continuous_columns = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol"
]
ordinal_columns = ["quality"]
nominal_columns = ["wine_type"]

variable_types = pd.DataFrame({
    "variavel": continuous_columns + ordinal_columns + nominal_columns,
    "tipo": ["continua"] * len(continuous_columns) + ["ordinal"] + ["nominal"]
})

display(variable_types)

## 5. Tabelas de frequencia

Para variaveis continuas, usa-se Sturges para definir o numero de classes. Para `quality` e `wine_type`, a distribuicao e simples, sem criar intervalos artificiais.

In [ ]:
def sturges_k(n: int) -> int:
    return math.ceil(1 + 3.322 * math.log10(n))


def rounded_class_width(series: pd.Series, k: int) -> float:
    raw_width = (series.max() - series.min()) / k
    if raw_width <= 0:
        return 1.0
    decimals = max(0, -math.floor(math.log10(raw_width)))
    factor = 10 ** decimals
    return math.ceil(raw_width * factor) / factor


def frequency_table_continuous(series: pd.Series) -> pd.DataFrame:
    clean = series.dropna().astype(float)
    n = len(clean)
    k = sturges_k(n)
    h = rounded_class_width(clean, k)
    start = clean.min()
    edges = [start + i * h for i in range(k + 1)]
    while edges[-1] <= clean.max():
        edges.append(edges[-1] + h)

    labels = [f"[{edges[i]:.6g}, {edges[i + 1]:.6g})" for i in range(len(edges) - 1)]
    classes = pd.cut(clean, bins=edges, right=False, labels=labels, include_lowest=True)
    fi = classes.value_counts(sort=False)

    table = pd.DataFrame({
        "classe_intervalo": labels,
        "xi_ponto_medio": [(edges[i] + edges[i + 1]) / 2 for i in range(len(edges) - 1)],
        "fi": fi.to_numpy(),
    })
    table["fr"] = table["fi"] / n
    table["Fi"] = table["fi"].cumsum()
    table["Fr"] = table["fr"].cumsum()
    table.insert(0, "variavel", series.name)
    table.insert(1, "n", n)
    table.insert(2, "min", clean.min())
    table.insert(3, "max", clean.max())
    table.insert(4, "amplitude_total", clean.max() - clean.min())
    table.insert(5, "k_sturges", k)
    table.insert(6, "h_amplitude_classe", h)
    return table


def frequency_table_categorical(series: pd.Series) -> pd.DataFrame:
    counts = series.value_counts(dropna=False).sort_index()
    table = counts.rename("fi").reset_index().rename(columns={"index": "valor"})
    table["fr"] = table["fi"] / table["fi"].sum()
    table["Fi"] = table["fi"].cumsum()
    table["Fr"] = table["fr"].cumsum()
    table.insert(0, "variavel", series.name)
    return table


freq_tables = {col: frequency_table_continuous(df[col]) for col in continuous_columns}
freq_tables["quality"] = frequency_table_categorical(df["quality"])
freq_tables["wine_type"] = frequency_table_categorical(df["wine_type"])

display(freq_tables["alcohol"])
display(freq_tables["quality"])

## 6. Exportacao opcional

Esta celula consolida as tabelas longas e salva uma copia atualizada no diretorio `export/`.

In [ ]:
freq_all = pd.concat(freq_tables.values(), ignore_index=True, sort=False)
df.to_csv(EXPORT_DIR / "wine_quality_original_from_base_notebook.csv", index=False)
freq_all.to_csv(EXPORT_DIR / "freq_all_variables_from_base_notebook.csv", index=False)

print("Arquivos salvos em:")
print(EXPORT_DIR / "wine_quality_original_from_base_notebook.csv")
print(EXPORT_DIR / "freq_all_variables_from_base_notebook.csv")

## 7. Visualizacoes principais

A variavel `alcohol` e uma boa candidata para graficos porque aparece na literatura como relevante para explicar qualidade.

In [ ]:
alcohol_freq = freq_tables["alcohol"].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=df, x="alcohol", hue="wine_type", bins=sturges_k(len(df)), kde=True, ax=axes[0])
axes[0].set_title("Alcohol por tipo de vinho")

sns.boxplot(data=df, x="wine_type", y="alcohol", ax=axes[1])
axes[1].set_title("Boxplot de alcohol")

axes[2].plot(alcohol_freq["xi_ponto_medio"], alcohol_freq["Fr"], marker="o")
axes[2].set_title("Ogiva - alcohol")
axes[2].set_xlabel("Ponto medio da classe")
axes[2].set_ylabel("Frequencia relativa acumulada")
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 8. Perguntas que este caderno ajuda a responder

- Quantas observacoes existem por tipo de vinho?
- Quais variaveis sao continuas, ordinal e nominal?
- Como construir distribuicoes de frequencia por Sturges?
- Qual a distribuicao da qualidade dos vinhos?
- Como `alcohol` se comporta por tipo de vinho?
- Quais variaveis fisico-quimicas podem ser comparadas com `quality`?